In [0]:
# Databricks notebook source
# COMMAND ----------
# MAGIC %md
# MAGIC # Task 5: Pipeline Quality Gate Check
# MAGIC Validates data completeness, reconciliation between layers, orphan records, and primary key uniqueness (accounting for SCD Type 2 history).

# COMMAND ----------
# Safe configuration retriever that doesn't trigger CONFIG_NOT_AVAILABLE
def get_spark_conf(key, default_val):
    try:
        return spark.conf.get(key)
    except Exception:
        return default_val

# Look for widget/parameter first, then spark conf, falling back safely to meridian_t12
try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = get_spark_conf("catalog_name", get_spark_conf("catalog", "meridian_t12"))

silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
ops_schema = f"{catalog}.ops"

print(f"Running quality gate against catalog: {catalog}")
catalog = spark.conf.get("catalog_name", spark.conf.get("catalog", "meridian_t12"))
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
ops_schema = f"{catalog}.ops"

errors = []
checks_passed = True

# Helper to log quality results to ops table
def log_quality_check(check_name, status, details):
    details_escaped = str(details).replace("'", "''")
    spark.sql(f"""
        INSERT INTO {ops_schema}.pipeline_quality_log 
        VALUES (current_timestamp(), '{check_name}', '{status}', '{details_escaped}')
    """)

# Ensure quality log table exists
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {ops_schema}.pipeline_quality_log (
        check_timestamp TIMESTAMP,
        check_name STRING,
        status STRING,
        details STRING
    )
""")

# COMMAND ----------
# MAGIC %md
# MAGIC ### 1. Primary Key Uniqueness Checks

# COMMAND ----------
# Standard SCD Type 1 Tables: Must have 0 duplicates across the entire table
scd1_tables = {
    f"{silver_schema}.silver_users": "user_id",
    f"{silver_schema}.silver_bookings": "booking_id",
    f"{silver_schema}.silver_payments": "payment_id",
    f"{silver_schema}.silver_destinations": "destination_id",
    f"{silver_schema}.silver_hosts": "host_id"
}

for table, pk in scd1_tables.items():
    try:
        dup_count = spark.sql(f"""
            SELECT count(*) - count(distinct {pk}) AS dup_count 
            FROM {table}
        """).collect()[0]["dup_count"]
        
        if dup_count > 0:
            msg = f"{table} has {dup_count} duplicate {pk} values"
            errors.append(msg)
            checks_passed = False
            log_quality_check(f"PK_CHECK_{table}", "FAIL", msg)
        else:
            log_quality_check(f"PK_CHECK_{table}", "PASS", f"0 duplicates on {pk}")
    except Exception as e:
        msg = f"Failed to check PK on {table}: {str(e)}"
        errors.append(msg)
        checks_passed = False
        log_quality_check(f"PK_CHECK_{table}", "ERROR", msg)

# SCD Type 2 Tables (Properties): Uniqueness check MUST be scoped to active records (__is_current = true)
try:
    prop_active_dups = spark.sql(f"""
        SELECT count(*) - count(distinct property_id) AS dup_count 
        FROM {silver_schema}.silver_properties 
        WHERE __END_AT IS NULL
    """).collect()[0]["dup_count"]

    if prop_active_dups > 0:
        msg = f"{silver_schema}.silver_properties has {prop_active_dups} duplicate active property_id values"
        errors.append(msg)
        checks_passed = False
        log_quality_check("PK_CHECK_silver_properties_scd2", "FAIL", msg)
    else:
        log_quality_check("PK_CHECK_silver_properties_scd2", "PASS", "0 active duplicates (SCD Type 2 compliant)")
except Exception as e:
    msg = f"Failed to check SCD Type 2 PK on silver_properties: {str(e)}"
    errors.append(msg)
    checks_passed = False
    log_quality_check("PK_CHECK_silver_properties_scd2", "ERROR", msg)

# COMMAND ----------
# MAGIC %md
# MAGIC ### 2. Orphan Row & Referential Integrity Checks

# COMMAND ----------
# Check for orphaned bookings pointing to non-existent properties
try:
    orphan_bookings_properties = spark.sql(f"""
        SELECT count(*) AS orphan_count 
        FROM {gold_schema}.gold_fact_bookings b
        LEFT JOIN {gold_schema}.gold_dim_properties p 
            ON b.property_id = p.property_id
        WHERE p.property_id IS NULL AND b.property_id IS NOT NULL
    """).collect()[0]["orphan_count"]

    if orphan_bookings_properties > 0:
        msg = f"Found {orphan_bookings_properties} orphaned bookings missing properties dimension"
        errors.append(msg)
        checks_passed = False
        log_quality_check("ORPHAN_ROW_CHECK_PROPERTIES", "FAIL", msg)
    else:
        log_quality_check("ORPHAN_ROW_CHECK_PROPERTIES", "PASS", "Zero orphaned bookings against properties")
except Exception as e:
    msg = f"Orphan check on properties failed: {str(e)}"
    errors.append(msg)
    checks_passed = False
    log_quality_check("ORPHAN_ROW_CHECK_PROPERTIES", "ERROR", msg)

# Check for orphaned bookings pointing to non-existent destinations
try:
    orphan_bookings_destinations = spark.sql(f"""
        SELECT count(*) AS orphan_count 
        FROM {gold_schema}.gold_fact_bookings b
        LEFT JOIN {gold_schema}.gold_dim_destinations d 
            ON b.destination_id = d.destination_id
        WHERE d.destination_id IS NULL AND b.destination_id IS NOT NULL
    """).collect()[0]["orphan_count"]

    if orphan_bookings_destinations > 0:
        msg = f"Found {orphan_bookings_destinations} orphaned bookings missing destination dimension"
        errors.append(msg)
        checks_passed = False
        log_quality_check("ORPHAN_ROW_CHECK_DESTINATIONS", "FAIL", msg)
    else:
        log_quality_check("ORPHAN_ROW_CHECK_DESTINATIONS", "PASS", "Zero orphaned bookings against destinations")
except Exception as e:
    msg = f"Orphan check on destinations failed: {str(e)}"
    errors.append(msg)
    checks_passed = False
    log_quality_check("ORPHAN_ROW_CHECK_DESTINATIONS", "ERROR", msg)

# COMMAND ----------
# MAGIC %md
# MAGIC ### 3. Table Non-Empty Completeness Check

# COMMAND ----------
required_tables = [
    f"{silver_schema}.silver_bookings",
    f"{silver_schema}.silver_properties",
    f"{gold_schema}.gold_fact_bookings",
    f"{gold_schema}.gold_dim_properties",
    f"{gold_schema}.gold_dim_destinations"
]

for table in required_tables:
    try:
        row_cnt = spark.table(table).count()
        if row_cnt == 0:
            msg = f"Table {table} is empty (0 rows)"
            errors.append(msg)
            checks_passed = False
            log_quality_check(f"COMPLETENESS_{table}", "FAIL", msg)
        else:
            log_quality_check(f"COMPLETENESS_{table}", "PASS", f"Row count: {row_cnt}")
    except Exception as e:
        msg = f"Failed to check row count on {table}: {str(e)}"
        errors.append(msg)
        checks_passed = False
        log_quality_check(f"COMPLETENESS_{table}", "ERROR", msg)

# COMMAND ----------
# MAGIC %md
# MAGIC ### 4. Quality Gate Final Evaluation

# COMMAND ----------
if not checks_passed:
    error_summary = " | ".join(errors)
    raise ValueError(f"Quality Gate Failed: {error_summary}")

print("All Quality Checks Passed Successfully.")